In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
import matplotlib.pyplot as plt



# duomenu ikelimas
data = pd.read_csv("CC GENERAL.csv")
data = data.drop('CUST_ID', axis=1)
data.head()



#  pradinė statistika (parodyti trūkstamas reikšmes, kardinalumą ir t. t.)
con_stats = []
for feature in data.columns:
    stat = {
        'Feature': feature,
        'Amount': len(data[feature]),
        'Missing %': (data[feature].isna().sum() / len(data[feature])) * 100,
        'Cardinality': data[feature].nunique(),
        'Max value': data[feature].max(),
        'Min value': data[feature].min(),
        'First quartile': data[feature].quantile(0.25),
        'Third quartile': data[feature].quantile(0.75),
        'Mean': data[feature].mean(),
        'Median': data[feature].median(),
        'Standard deviation': data[feature].std()
    }
    con_stats.append(stat)

con_stats_df = pd.DataFrame(con_stats).set_index('Feature')
con_stats_df



#  trūkstamų reikšmių ir kategorinių/„frequency" kintamųjų šalinimas
data = data.dropna()
data = data.drop(['BALANCE_FREQUENCY',
                  'PURCHASES_FREQUENCY',
                  'ONEOFF_PURCHASES_FREQUENCY',
                  'PURCHASES_INSTALLMENTS_FREQUENCY',
                  'PRC_FULL_PAYMENT',
                  'TENURE'], axis=1)



# ekstremalių reikšmių šalinimas (IQR), tik tolydinių (≥50 kategorijų) atranka ir standartizavimas
Q1 = data.quantile(0.25)
Q3 = data.quantile(0.75)
IQR = Q3 - Q1
data_clean = data[~((data < (Q1 - 1.5 * IQR)) | (data > (Q3 + 1.5 * IQR))).any(axis=1)]

continuous_cols = [col for col in data_clean.columns if data_clean[col].nunique() >= 50]
data_cont = data_clean[continuous_cols]

scaler = StandardScaler()
data_scaled = pd.DataFrame(scaler.fit_transform(data_cont), columns=continuous_cols)
data_scaled.head()


,BALANCE,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS
0,-0.805690,-0.699902,-0.604747,-0.382513,-0.561305,-0.897822,-0.822601,-0.651558
1,1.634862,0.800192,1.607593,-0.729293,-0.561305,1.679603,-0.236043,0.803874
2,-0.033211,-0.875636,-0.558965,-0.729293,-0.561305,-0.818517,-0.157510,-0.337418
3,0.967113,0.054384,-0.604747,0.856295,-0.561305,-0.382337,-0.156491,0.519663
4,0.162900,0.995669,1.288033,-0.002292,-0.561305,1.481339,-0.143632,-0.136988


In [ ]:
def winnerMc(x, weights):
    """KONKURENCIJA. Randa BMU (Best Matching Unit) m_c — artimiausią mazgą
       pagal Euklido atstumą:   c(x) = argmin_i || x - w_i ||"""
    dists = np.linalg.norm(weights - x, axis=1)      # Euklido atstumas iki kiekvieno mazgo
    return int(np.argmin(dists))

def radius(t, T, sigma0):
    """Kaimynystės SPINDULYS sigma(t). Iš pradžių didelis (apima daug kaimynų),
       su kiekviena iteracija mažėja eksponentiškai:
            sigma(t) = sigma0 * exp(-t / lambda),   lambda = T / ln(sigma0)"""
    lam = T / np.log(sigma0) if sigma0 > 1 else T
    return sigma0 * np.exp(-t / lam)

def learning_rate(t, T, alpha0):
    """Mokymosi greitis alpha(t) ∈ (0, 1]. Mažėja eksponentiškai:
            alpha(t) = alpha0 * exp(-t / T)"""
    return alpha0 * np.exp(-t / T)

def neighbourhood(c, k, sigma):
    """Gauso kaimynystės (BRANDUOLIO) funkcija h_{i,c}  — formulė (8.16).
       1D gardelėje atstumas tarp mazgų i ir c (= BMU) yra |i - c|:
            h_{i,c} = exp( -|i - c|^2 / (2 * sigma^2) )
       Pirmosios eilės kaimynai: |i-c|=1, antrosios: |i-c|=2, ..."""
    idx = np.arange(k)
    return np.exp(-((idx - c) ** 2) / (2.0 * sigma ** 2))

def rewardNeighbours(x, weights, c, alpha, sigma):
    """PRITAIKYMAS — kaimynų 'apdovanojimas', formulė (8.15).
       BMU ir jo kaimynai pastumiami link x; kuo arčiau BMU (didesnis h),
       tuo labiau keičiami svoriai (stipresnis ryšys):
            w_i <- w_i + alpha * h_{i,c} * (x - w_i)         (visiems i)"""
    k = weights.shape[0]
    h = neighbourhood(c, k, sigma)                   # (k,) — ryšio stiprumas kiekvienam mazgui
    weights += (alpha * h)[:, None] * (x - weights)  # in-place atnaujinimas
    return weights


def SOM(X, k, n_iter=5000, alpha0=0.5, random_state=42):
    """1D SOM su k mazgų (= k klasterių).
       n_iter — iteracijų skaičius (rekom. 1000–8000).
       Grąžina: weights (k×d) ir labels (n,) — kiekvieno taško klasteris."""
    rng = np.random.default_rng(random_state)
    X = np.asarray(X, dtype=float)
    n, d = X.shape
    # --- Svorių INICIALIZACIJA: k atsitiktinai parinktų duomenų taškų ---
    weights = X[rng.choice(n, size=k, replace=False)].copy()
    sigma0 = max(1.0, k / 2.0)                        # pradinis spindulys ≈ pusė gardelės
    # --- MOKYMAS (online) ---
    for t in range(n_iter):
        x = X[rng.integers(n)]                        # atsitiktinis imties elementas
        c = winnerMc(x, weights)                      # konkurencija (BMU)
        a = learning_rate(t, n_iter, alpha0)
        s = radius(t, n_iter, sigma0)
        rewardNeighbours(x, weights, c, a, s)    
    # --- Klasterių ŽYMĖS = kiekvieno taško BMU ---
    dists = np.linalg.norm(X[:, None, :] - weights[None, :, :], axis=2)   # (n, k)
    labels = np.argmin(dists, axis=1)
    return weights, labels        


In [9]:
def inertia(X, weights, labels):
    """INERCIJA = SOM kvantavimo PAKLAIDA  E_Q = Σ_j || x_j - w_{BMU(j)} ||^2.
       Tai suma kvadratinių atstumų nuo kiekvieno taško iki jo klasterio centro
       (čia centras = BMU mazgo svorių vektorius).
       Maža inercija → taškai arti savo centrų → 'tankūs' klasteriai.
       Visada mažėja didinant k → naudojama 'alkūnės' (elbow) metodui."""
    X = np.asarray(X, dtype=float)
    return float(np.sum(np.linalg.norm(X - weights[labels], axis=1) ** 2))


def silhouette(X, labels):
    """SILUETO KOEFICIENTAS ∈ [-1, 1]. Kiekvienam taškui:
            s = (b - a) / max(a, b)
       a = vid. atstumas iki SAVO klasterio taškų (kohezija),
       b = vid. atstumas iki artimiausio KITO klasterio taškų (separacija).
       Visumos siluetas = visų taškų s vidurkis.
        s ≈  1  → gerai atskirti, tankūs klasteriai
        s ≈  0  → taškai ant klasterių ribos
        s <  0  → taškai greičiausiai priskirti blogam klasteriui
       Reikia ≥ 2 unikalių žymių (kitaip neapibrėžtas)."""
    if len(np.unique(labels)) < 2:
        return np.nan
    return float(silhouette_score(X, labels))

In [10]:
X2 = data_scaled[['BALANCE', 'PURCHASES']].values
w, lab = SOM(X2, k=3, n_iter=5000)
print("Klasteriai:", np.unique(lab))
print("Inercija (E_Q):", inertia(X2, w, lab))
print("Siluetas:", silhouette(X2, lab))

Klasteriai: [0 1 2]
Inercija (E_Q): 7324.805925835747
Siluetas: 0.2535659499679282


Reikalaujama: lentelė + scatter grafikai + inercijos grafikai + silueto diagramos + clustergram. 

# https://www.geeksforgeeks.org/python/self-organising-maps-kohonen-maps/


Tinklelio struktūra -  1D tiesinė gardelė su k mazgų.

Svorių inicializacija - atsitiktinai parinktų elementų inicializacija

Atstumo formulė (BMU paieška) - Euklido atstumas

skaičiuojamas BMU kaimynystės spindulys. Šalia
neurono 𝑚𝑐 esantys neuronai vadinami pirmosios eilės kaimynai, paskui antrosios ir t. t. Iš pradžių spindulys būna didesnis, o paskui su kiekviena iteracija mažėja, naudojama formule: Kaimynystės funkcija - branduolys (8.16) [Gauso]

toliau kaimynai yra “apdovanojami”, Kuo viršūnė yra arčiau neurono 𝑚𝑐
, tuo labiau keičiami jos svoriai – tuo labiau “apdovanojami”. Tai reiškiasi, kad
ryšys yra stipresnis. naudojama formule (8.15)



mokymosi greitis - [0, 1]
Sustojimo sąlyga - epochs = [100,500] iterations= [1000,8000] 
Mc - BMU mazgas, inercija- ? nezinau kas tai, silueto koef - irgi nezinau

paklaida reikia skaiciuoti !!!!!!!!!!!!!!







Kas yra inercija? Suma kvadratinių atstumų nuo kiekvieno taško iki jo klasterio centro. SOM atveju centras = to taško BMU mazgo svorių vektorius — todėl inercija sutampa su kvantavimo paklaida $E_Q$ iš 11_SOM.pdf 10 skaidrės. Mažesnė inercija = tankesni klasteriai. Ji visada mažėja didinant k, todėl pati savaime „geriausio k" nepasako — bet braižant inercija vs k ieškoma „alkūnės" taško (kur kreivė staigiai išsilygina).

Kas yra silueto koeficientas? Matas, kaip gerai taškas „tinka" savo klasteryje: $s=(b-a)/\max(a,b)$, kur $a$ — vidutinis atstumas iki savo klasterio kaimynų, $b$ — iki artimiausio svetimo klasterio. $s\in[-1,1]$, didesnis = geriau atskirti klasteriai. Viso klasterizavimo siluetas = visų taškų $s$ vidurkis (silhouette_score). Šitą metriką ir naudoji rinkdamasis geriausią k bei geriausias atributų poras. Silueto diagramoms (3 uždavinys) naudosi silhouette_samples(X, labels) — gausi $s$ kiekvienam taškui atskirai.

Apie n_iter: užduočiai užtenka 5000 (konspektas mini „>1000 iteracijų"). Testuodamas gali sumažinti iki 1000, kad greičiau suktųsi; galutiniam rezultatui — 5000–8000.

Kitas žingsnis — ciklas per visas 28 poras × k=2..8, kuris pripildo lentelę ir išrenka 3 geriausias poras grafikams. Nori, kad ir tą parašyčiau?